In [1]:
# Step 1：Import and read the data
import torch
import torch.nn as nn
import torch.optim as optim
import pickle

from dictionary_learning import AutoEncoder
from dictionary_learning.trainers import StandardTrainer
from dictionary_learning.training import trainSAE

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print(f"Using device: {device}")

with open('train_activation_matrix.pkl', 'rb') as instream:
    train_data = pickle.load(instream)
with open('eval_activation_matrix.pkl', 'rb') as instream:
    eval_data = pickle.load(instream)
with open('all_activation_matrix.pkl', 'rb') as instream:
    all_data = pickle.load(instream)

print(f"train: {train_data.shape}, eval: {eval_data.shape}, all: {all_data.shape}")

Using device: cpu
train: torch.Size([145444, 128]), eval: torch.Size([16160, 128]), all: torch.Size([161604, 128])


In [ ]:
#Pre_Step: Check Activation Statistics
print("all_data stats:")
print(f"  mean : {all_data.mean().item():.4f}")
print(f"  std  : {all_data.std().item():.4f}")
print(f"  min  : {all_data.min().item():.4f}")
print(f"  max  : {all_data.max().item():.4f}")
print(f"  % values > 0 : {(all_data > 0).float().mean().item()*100:.1f}%")

all_data stats:
  mean : 1.7347
  std  : 7.4440
  min  : 0.0000
  max  : 659.5259
  % values > 0 : 45.5%


In [ ]:
# Cell 1.5: Normalize activations for SAE 
# Step 1: center — subtract per-feature mean (shape: 1×128)
mean_act = all_data.mean(dim=0, keepdim=True)
train_data_c = train_data - mean_act
eval_data_c  = eval_data  - mean_act
all_data_c   = all_data   - mean_act

# Step 2: scale — divide by 95th percentile of L2 norms (robust to outliers)
norms = train_data_c.norm(dim=1)
scale = torch.quantile(norms, 0.95).item()
train_data_n = train_data_c / scale
eval_data_n  = eval_data_c  / scale
all_data_n   = all_data_c   / scale

# Save for later inference
torch.save({"mean_act": mean_act, "scale": scale},
           os.path.join(PKL_DIR, "activation_norm_params.pt"))

print(f"mean_act shape: {mean_act.shape}, scale: {scale:.4f}")
print(f"After norm — mean: {all_data_n.mean().item():.4f}, "
      f"std: {all_data_n.std().item():.4f}, "
      f"max: {all_data_n.max().item():.4f}")

# Before normalization: mean=1.73, std=7.44, max=659
# After normalization:  mean≈0,   std=0.041, max=3.70

mean_act shape: torch.Size([1, 128]), scale: 177.4239
After norm — mean: -0.0000, std: 0.0410, max: 3.6988


In [ ]:
#Cell 2: Train SAE on normalized activations ───────────────────────
import os, shutil, torch
from torch.utils.data import DataLoader
from dictionary_learning import AutoEncoder
from dictionary_learning.trainers import StandardTrainer
from dictionary_learning.training import trainSAE

PKL_DIR  = "/Users/elvainyu/Desktop/申请与学习/暑研和实习/暑研/‼️2026暑研资料/Computational Biology/Papers & Datas/SAE training/sae_code"
SAVE_DIR = os.path.join(PKL_DIR, "sae_dictionary-learning/")

# Ablation: only change this line each round ----------------------------------------
L1_PENALTY = 5e-4   # Final selected value; ablation candidates: 1e-3, 1e-4, 5e-3
# -------------------------------------------------------------------------------

backup_name = f"sae_backup_normed_l1pen{L1_PENALTY}/"
BACKUP = os.path.join(PKL_DIR, backup_name)
if os.path.exists(SAVE_DIR) and not os.path.exists(BACKUP):
    shutil.copytree(SAVE_DIR, BACKUP)
    print(f"Backed up to: {backup_name}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Use normalized data
n_epoch = 20
train_rep = train_data_n.repeat((n_epoch, 1))
train_loader = DataLoader(train_rep, batch_size=64, shuffle=True)
total_steps = len(train_loader)
print(f"Total steps: {total_steps}")

trainer_cfg = {
    "trainer": StandardTrainer,
    "dict_class": AutoEncoder,
    "activation_dim": 128,
    "dict_size": 2048,
    "lr": 1e-3,
    "l1_penalty": L1_PENALTY,
    "steps": total_steps,
    "warmup_steps": 1000,
    "sparsity_warmup_steps": 2000,
    "resample_steps": 3000,
    "layer": 1,
    "lm_name": "test",
    "device": str(device),
}

trainSAE(
    data=train_loader,
    trainer_configs=[trainer_cfg],
    steps=total_steps,
    save_dir=SAVE_DIR,
    verbose=True,
    log_steps=500,
)

# Diagnostics
ae_check = AutoEncoder.from_pretrained(os.path.join(SAVE_DIR, "trainer_0/ae.pt"))
ae_check.to(device).eval()

with torch.no_grad():
    recons_check, features_check = ae_check(all_data_n.to(device), output_features=True)

dead_mask = (features_check == 0).all(dim=0)
l0        = (features_check > 0).float().sum(dim=1).mean().item()
orig_var  = all_data_n.var(dim=0).mean().item()
res_var   = (all_data_n.to(device) - recons_check).var(dim=0).mean().item()
frac_var  = 1 - res_var / orig_var

print(f"\n{'='*40}")
print(f"l1_penalty    = {L1_PENALTY}")
print(f"Dead features : {dead_mask.sum().item()}/2048")
print(f"L0 norm       : {l0:.1f}  (target: 20–80)")
print(f"frac_var_expl : {frac_var:.4f}  (target: >0.5)")
print(f"{'='*40}")

Device: cpu
Total steps: 45452


  0%|          | 19/45452 [00:00<04:01, 187.79it/s]

Step 0: L0 = 1023.1875, frac_variance_explained = 0.14905494451522827


  1%|          | 544/45452 [00:01<02:03, 363.84it/s]

Step 500: L0 = 1038.5625, frac_variance_explained = 0.9984657168388367


...

Step 44500: L0 = 39.84375, frac_variance_explained = 0.9975714683532715


 99%|█████████▉| 45059/45452 [01:35<00:00, 484.17it/s]

Step 45000: L0 = 35.421875, frac_variance_explained = 0.9931235909461975
resampling 1879 neurons


100%|██████████| 45452/45452 [01:36<00:00, 469.40it/s]



l1_penalty    = 0.0005
Dead features : 1839/2048
L0 norm       : 41.6  (target: 20–80)
frac_var_expl : 0.9940  (target: >0.5)


In [ ]:

#  Cell 3: Extract features using normalized data 
import pickle, os, torch
from dictionary_learning import AutoEncoder

PKL_DIR  = "/Users/elvainyu/Desktop/申请与学习/暑研和实习/暑研/‼️2026暑研资料/Computational Biology/Papers & Datas/SAE training/sae_code"
SAVE_DIR = os.path.join(PKL_DIR, "sae_dictionary-learning/")
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ae = AutoEncoder.from_pretrained(os.path.join(SAVE_DIR, "trainer_0/ae.pt"))
ae.to(device).eval()

with torch.no_grad():
    recons_train, features_train = ae(train_data_n.to(device), output_features=True)
    recons_eval,  features_eval  = ae(eval_data_n.to(device),  output_features=True)
    recons_all,   features_all   = ae(all_data_n.to(device),   output_features=True)

recons_train  = recons_train.cpu()
features_train = features_train.cpu()
recons_eval   = recons_eval.cpu()
features_eval  = features_eval.cpu()
recons_all    = recons_all.cpu()
features_all   = features_all.cpu()

with open(os.path.join(PKL_DIR, "sae_results.pkl"), "wb") as f:
    pickle.dump((recons_train, features_train, recons_eval, features_eval), f)

with open(os.path.join(PKL_DIR, "sae_results_all.pkl"), "wb") as f:
    pickle.dump((recons_all, features_all), f)

print(f"features_train : {features_train.shape}")
print(f"features_eval  : {features_eval.shape}")
print(f"features_all   : {features_all.shape}")

features_train : torch.Size([145444, 2048])
features_eval  : torch.Size([16160, 2048])
features_all   : torch.Size([161604, 2048])


In [ ]:
#  Cell 3.5: Load AnnData 
import scanpy as sc

H5AD_PATH = "/Users/elvainyu/Desktop/申请与学习/暑研和实习/暑研/‼️2026暑研资料/Computational Biology/Papers & Datas/Papers' Dataset/BoneMarrowMap_Unsorted_scVI.h5ad"

adata_unsorted = sc.read_h5ad(H5AD_PATH)
print(f"Unsorted dataset: {adata_unsorted.shape}")
print(f"Sorting values  : {adata_unsorted.obs['Sorting'].unique().tolist()}")
print(f"CellType count  : {adata_unsorted.obs['CellType'].nunique()} types")

Unsorted dataset: (161604, 4000)
Sorting values  : ['Bulk']
CellType count  : 55 types


In [ ]:
# Cell 4: Write SAE features to adata.obsm + subset B cells 
import pickle, os
import numpy as np

PKL_DIR = "/Users/elvainyu/Desktop/申请与学习/暑研和实习/暑研/‼️2026暑研资料/Computational Biology/Papers & Datas/SAE training/sae_code"

with open(os.path.join(PKL_DIR, "sae_results_all.pkl"), "rb") as f:
    data = pickle.load(f)

features_np = data[1].detach().numpy()   # shape (161604, 2048)
print(f"SAE features shape: {features_np.shape}")

adata_unsorted.obsm["X_sae"] = features_np

B_CELL_TYPES = [
    "HSC", "LMPP", "MLP", "MLP-II", "CLP",
    "Pro-B Cycling", "Pro-B VDJ",
    "Large Pre-B", "Small Pre-B",
    "Immature B", "Mature B", "Plasma Cell"
]

mask = adata_unsorted.obs["CellType"].isin(B_CELL_TYPES)
adata_bcell = adata_unsorted[mask].copy()

print(f"Full dataset : {adata_unsorted.shape}")
print(f"B-cell subset: {adata_bcell.shape}")
print(f"Cell types   :\n{adata_bcell.obs['CellType'].value_counts()}")

SAE features shape: (161604, 2048)
Full dataset : (161604, 4000)
B-cell subset: (18201, 4000)
Cell types   :
CellType
Mature B         6197
Plasma Cell      2520
Large Pre-B      2499
Immature B       1569
Pro-B VDJ        1279
HSC              1116
Pro-B Cycling     867
Small Pre-B       759
LMPP              715
CLP               316
MLP               260
MLP-II            104
Name: count, dtype: int64


In [ ]:
#  Cell 5: ANOVA F-test — B-cell discriminative SAE features 
import numpy as np
from sklearn.feature_selection import f_classif

X = adata_unsorted.obsm["X_sae"]                          # (161604, 2048)
y = adata_unsorted.obs["CellType"].isin(B_CELL_TYPES).astype(int).values

# Check dead features
variances = X.var(axis=0)
dead_mask = variances == 0
print(f"Dead features: {dead_mask.sum()}/2048")
print(f"Active features: {(~dead_mask).sum()}/2048")

# ANOVA computed on active features only
X_active = X[:, ~dead_mask]
F_stats, p_vals = f_classif(X_active, y)

# Map back to original 2048-dim indices
active_idx = np.where(~dead_mask)[0]
F_full  = np.zeros(2048)
p_full  = np.ones(2048)
F_full[active_idx] = F_stats
p_full[active_idx] = p_vals

ranked_idx = np.argsort(F_full)[::-1]
top50_idx  = ranked_idx[:50]

print(f"\nTop 10 features:")
for i, idx in enumerate(ranked_idx[:10]):
    print(f"  Rank {i+1}: feature #{idx:4d}  F={F_full[idx]:.1f}  p={p_full[idx]:.2e}")

print(f"\nF-stat range — max: {F_full.max():.1f}, rank50: {F_full[top50_idx[-1]]:.1f}, min active: {F_stats.min():.1f}")

Dead features: 1839/2048
Active features: 209/2048

Top 10 features:
  Rank 1: feature # 126  F=21644.9  p=0.00e+00
  Rank 2: feature #   3  F=12262.4  p=0.00e+00
  Rank 3: feature #  63  F=12057.3  p=0.00e+00
  Rank 4: feature #  24  F=12043.1  p=0.00e+00
  Rank 5: feature #  16  F=11499.4  p=0.00e+00
  Rank 6: feature #  27  F=10803.8  p=0.00e+00
  Rank 7: feature #  71  F=10062.2  p=0.00e+00
  Rank 8: feature #  11  F=9006.4  p=0.00e+00
  Rank 9: feature # 314  F=8755.4  p=0.00e+00
  Rank 10: feature #  17  F=8439.1  p=0.00e+00

F-stat range — max: 21644.9, rank50: 1413.8, min active: 0.0


In [24]:
#  Cell 6: Stage 4C — Gene signatures for top SAE features 
import numpy as np
from scipy import stats

# Extract gene signatures for top features
TOP_N_FEATURES = 10
TOP_CELLS = 200      # Top-N most highly activated cells per feature
TOP_GENES  = 30      # Number of top genes to report per feature

X_sae   = adata_unsorted.obsm["X_sae"]          # (161604, 2048)
X_genes = adata_unsorted.X                       # (161604, 4000) gene expression matrix
gene_names = adata_unsorted.var_names.tolist()

import scipy.sparse as sp
if sp.issparse(X_genes):
    X_genes = X_genes.toarray()

results = {}

for rank, feat_idx in enumerate(ranked_idx[:TOP_N_FEATURES]):
    activations = X_sae[:, feat_idx]             # Feature activations across all cells

    # Find top-activated cells
    top_cell_idx = np.argsort(activations)[::-1][:TOP_CELLS]
    bg_cell_idx  = np.argsort(activations)[:(len(activations) - TOP_CELLS)]

    top_expr = X_genes[top_cell_idx, :]          # (200, 4000)
    bg_expr  = X_genes[bg_cell_idx,  :]          # (rest, 4000)

    # t-test: differential expression in top-activated vs background cells
    t_stats, p_vals = stats.ttest_ind(top_expr, bg_expr, axis=0, equal_var=False)
    logfc = np.log1p(top_expr.mean(axis=0)) - np.log1p(bg_expr.mean(axis=0))

    # Rank by t-stat (positive = higher expression in top-activated cells)
    gene_rank = np.argsort(t_stats)[::-1]
    top_gene_names = [gene_names[i] for i in gene_rank[:TOP_GENES]]
    top_gene_logfc = logfc[gene_rank[:TOP_GENES]]

    results[feat_idx] = {
        "rank": rank + 1,
        "F_stat": F_full[feat_idx],
        "top_genes": top_gene_names,
        "logfc": top_gene_logfc,
    }

    print(f"\nRank {rank+1} | Feature #{feat_idx} | F={F_full[feat_idx]:.1f}")
    print(f"  Top genes: {', '.join(top_gene_names[:15])}")


Rank 1 | Feature #126 | F=21644.9
  Top genes: STMN1, HMGN2, TUBB, TUBA1B, H2AFZ, HMGB1, RPS24, PPIA, HMGN1, RPLP0, PTMA, HNRNPA2B1, RPS2, RPS17, HNRNPA1

Rank 2 | Feature #3 | F=12262.4
  Top genes: SSR4, IGKC, FKBP11, SEC11C, ITM2C, PRDX4, XBP1, MZB1, ERLEC1, PLPP5, HERPUD1, GNG7, CRELD2, ISG20, SDF2L1

Rank 3 | Feature #63 | F=12057.3
  Top genes: SSR4, IGKC, PRDX4, FKBP11, SEC11C, XBP1, ITM2C, PLPP5, MZB1, FKBP2, HERPUD1, ERLEC1, GNG7, FCRL5, CD63

Rank 4 | Feature #24 | F=12043.1
  Top genes: SSR4, IGKC, PRDX4, FKBP11, SEC11C, ITM2C, XBP1, PDIA6, MZB1, DAD1, MYDGF, FKBP2, ERLEC1, HERPUD1, PLPP5

Rank 5 | Feature #16 | F=11499.4
  Top genes: IGKC, SSR4, PRDX4, FKBP11, SEC11C, ITM2C, XBP1, MZB1, PDIA6, FKBP2, MYDGF, ERLEC1, DAD1, HERPUD1, PLPP5

Rank 6 | Feature #27 | F=10803.8
  Top genes: SSR4, IGLC2, SEC11C, FKBP11, MZB1, ITM2C, PRDX4, PLPP5, XBP1, HERPUD1, FKBP2, CRELD2, GNG7, ERLEC1, SDF2L1

Rank 7 | Feature #71 | F=10062.2
  Top genes: SSR4, IGKC, PRDX4, FKBP11, SEC11C, ITM2C